**Como estão a conversão e o volume de propostas comerciais por região e por status — de quantas propostas viram negócio fechado, onde estão concentradas geograficamente, e qual o ticket médio por região**

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.silver;

CREATE OR REPLACE TABLE workspace.silver.proposal AS
SELECT
    id,
    fk_requester,
    status,
    total_amount,
    created_at,
    updated_at
FROM solaria.public.proposal
WHERE total_amount IS NOT NULL
  AND total_amount > 0
QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY updated_at DESC) = 1;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.gold;

CREATE OR REPLACE TABLE workspace.gold.dim_tempo AS
SELECT DISTINCT
    CAST(date_trunc('day', created_at) AS DATE) AS data_id,
    YEAR(created_at)  AS ano,
    MONTH(created_at) AS mes,
    QUARTER(created_at) AS trimestre,
    DAYOFWEEK(created_at) AS dia_semana
FROM workspace.silver.proposal;

CREATE OR REPLACE TABLE workspace.gold.dim_regiao AS
SELECT DISTINCT
    a.id     AS regiao_id,
    a.state  AS estado,
    a.city   AS cidade
FROM solaria.public.requester r
JOIN solaria.public.company c ON c.id = r.fk_company
JOIN solaria.public.address a ON a.id = c.fk_address;

CREATE OR REPLACE TABLE workspace.gold.dim_cliente AS
SELECT
    r.id AS cliente_id,
    r.fk_company,
    c.trade_name,
    c.fk_address AS regiao_id
FROM solaria.public.requester r
JOIN solaria.public.company c ON c.id = r.fk_company;

CREATE OR REPLACE TABLE workspace.gold.fato_propostas AS
SELECT
    p.id AS proposta_id,
    p.fk_requester AS cliente_id,
    dc.regiao_id,
    CAST(date_trunc('day', p.created_at) AS DATE) AS data_id,
    p.status,
    p.total_amount
FROM workspace.silver.proposal p
JOIN workspace.gold.dim_cliente dc ON dc.cliente_id = p.fk_requester;

CREATE OR REPLACE TABLE workspace.gold.dim_regiao_geo AS
SELECT DISTINCT
    lu.id AS local_unit_id,
    a.state AS estado,
    a.city AS cidade,
    g.latitude,
    g.longitude
FROM solaria.public.local_unit lu
JOIN solaria.public.address a ON a.id = lu.fk_address
JOIN solaria.public.geolocalization g ON g.fk_address = a.id;

CREATE OR REPLACE TABLE workspace.gold.fato_propostas_geo AS
SELECT
    pu.fk_proposal_item,
    pi.fk_proposal AS proposta_id,
    dg.latitude,
    dg.longitude
FROM solaria.public.proposal_unit pu
JOIN solaria.public.proposal_item pi ON pi.id = pu.fk_proposal_item
JOIN workspace.gold.dim_regiao_geo dg ON dg.local_unit_id = pu.fk_local_unit;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.kpi_regiao AS
SELECT
    dr.estado,
    dr.cidade,
    COUNT(*) AS total_propostas,
    SUM(f.total_amount) AS valor_total,
    AVG(f.total_amount) AS valor_medio,
    STDDEV(f.total_amount) AS desvio_valor,
    RANK() OVER (ORDER BY SUM(f.total_amount) DESC) AS ranking_valor
FROM workspace.gold.fato_propostas f
JOIN workspace.gold.dim_regiao dr ON dr.regiao_id = f.regiao_id
GROUP BY dr.estado, dr.cidade;

CREATE OR REPLACE TABLE workspace.gold.kpi_conversao_status AS
SELECT
    status,
    COUNT(*) AS quantidade,
    COUNT(*) * 100.0 / SUM(COUNT(*)) OVER () AS percentual
FROM workspace.gold.fato_propostas
GROUP BY status;

CREATE OR REPLACE TABLE workspace.gold.kpi_evolucao_mensal AS
SELECT
    dt.ano, dt.mes,
    COUNT(*) AS propostas_no_mes,
    SUM(COUNT(*)) OVER (ORDER BY dt.ano, dt.mes) AS running_total
FROM workspace.gold.fato_propostas f
JOIN workspace.gold.dim_tempo dt ON dt.data_id = f.data_id
GROUP BY dt.ano, dt.mes;